# Transaction Foundation Model on Ray — Part 5: Batch embedding extraction

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: ~15 min on the GPU workers)

---

Previously in Part 4, we trained the foundation model. With the foundation model built, now we build the first fraud detector. 

In this first approach, we will use **embeddings** — the model's vector representation of a transaction. This notebook produces the embeddings, and Part 6 trains on them and evaluates the results against the baselines from NVIDIA.

Later, Part 7 builds a stronger detector by fine-tuning the foundation model itself.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import logging

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},
         logging_level=logging.ERROR)

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = every card
cfg = load_scale(SCALE)
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Turn each transaction into an embedding

We embed each transaction on its own, following NVIDIA's blueprint. The transaction goes through the model as a 14-token sequence — its 12 field tokens (amount, location, etc) covered in Part 3. The model's internal state at the last token is the embedding: the fixed-length vector that Part 6's fraud detectors will train on.

The vector re-encodes only the transaction's own 13 fields, enriched by what the model learned about transactions in general during pretraining. The card's full transaction history is not included. This matches NVIDIA's approach. Part 7 revisits that choice by including history windows.

We embed to create three datasets:

- **train** — NVIDIA's balance-sampling recipe: every fraud transaction, plus normal ones up to 1M rows at `full`. This keeps fraud frequent enough to learn from and cuts the embedding work from 19.5M transactions to about 1M.
- **val** and **test** — the 100K stratified eval subsets from Part 2, embedded whole.

## Stream CPU work into GPU forward passes

One Ray Data pipeline runs this whole stage on two kinds of hardware at once. Tokenizing transactions is CPU work, the forward pass is GPU work, and the model is expensive to load — it should load once per GPU and stay in memory. The pipeline gives each piece its own resources:

- One **CPU task** per split prepares the sample and writes the labels and raw features straight to storage — that part never needs a GPU.
- **CPU workers** turn the transactions into tokens, in batches.
- **GPU actors** run the model. Each actor loads the model once as a Ray Actor, then batches stream through. This is the typical Ray Data pattern for GPU inference.

The pools scale independently. More CPU workers keep the GPUs busy, more GPU actors raise throughput, and each pool uses only its own hardware. Every row is numbered with its output position, so the results come out in the same order as the single-GPU reference.

### Run the embedding pipeline

One loop runs the three splits.

In [2]:
from src.nvembed import prepare_embed_split, encode_txn_batch, assemble_embed_task, wait_for_files
import shutil

# The GPU actor. Its constructor runs once per actor: it loads the trained model
# with NVIDIA's inference wrapper, set to read the internal state at the last
# token. Every call after that embeds one batch.
class GPUEmbedder:
    def __init__(self, hf_dir: str, merchant_hash: int = 2000, batch_size: int = 1024):
        import os
        import sys
        # This torch build routes some ops (bmm) through triton JIT, which needs a C
        # compiler the GPU workers don't have — fall back to the standard kernels.
        os.environ.setdefault("TORCH_DISABLE_NATIVE_JIT", "1")
        sys.path.insert(0, ".")
        from src.decoder_inference import HuggingFaceDecoderInference
        from src.nvidia_tokenizer import FinancialTabularTokenizer

        tok = FinancialTabularTokenizer(merchant_hash_size=merchant_hash,
                                        category_hierarchy=True, temporal_encoding=True)
        self.inf = HuggingFaceDecoderInference(model_path=hf_dir, tokenizer=tok,
                                               pooling="last_token")
        self.batch_size = batch_size

    def __call__(self, batch):
        emb = self.inf.extract_embeddings_batched(batch["ids"], batch_size=self.batch_size,
                                                  show_progress=False)
        return {"__pos__": batch["__pos__"], "emb": emb.astype("float32")}


eb = cfg["embed"]
use_gpu = eb["use_gpu"]        # mini runs the actors on CPU; full gives each actor a GPU

if not os.path.exists(os.path.join(paths["embeddings"], "embed_test.npy")):
    # train is sampled from the training data; val and test are the 100K
    # evaluation files from Part 2.
    for split, fname in {"train": None, "val": "val_eval.parquet",
                         "test": "test_eval.parquet"}.items():
        
        # prepare_embed_split picks this split's rows and writes their labels and
        # raw features. It runs as one Ray task on a worker.
        prep = ray.get(prepare_embed_split.remote(
            paths["nvsplit"], fname, paths["embeddings"], split,
            eb["balanced_train"], 42))

        shards = os.path.join(paths["embeddings"], f"_emb_{split}")
        (ray.data.read_parquet(prep["prep"])
            # encode_txn_batch turns a batch of transactions into token rows. It is
            # plain pandas and runs on the CPU workers.
            .map_batches(encode_txn_batch, batch_format="pandas")
            # GPUEmbedder is a class, so map_batches builds an actor pool: each actor
            # loads the model once in its constructor, then embeds batch after batch.
            .map_batches(GPUEmbedder,
                         fn_constructor_kwargs={"hf_dir": paths["hf"],
                                                "batch_size": eb["batch_size"]},
                         batch_format="numpy",
                         batch_size=4096,    # rows per actor call; the GPU runs eb["batch_size"] at a time
                         num_gpus=(1 if use_gpu else 0),
                         concurrency=eb["num_workers"])
            .write_parquet(shards))

        # assemble_embed_task gathers the shards into embed_<split>.npy in row
        # order. It runs as one Ray task.
        out_path = os.path.join(paths["embeddings"], f"embed_{split}.npy")
        meta = ray.get(assemble_embed_task.remote(shards, out_path, cfg["model"]["d_model"]))
        wait_for_files([out_path])   # shared storage: a worker's write can lag the driver's view
        shutil.rmtree(shards)        # the shards were only inputs to the assembly
        if os.path.exists(prep["prep"]):
            os.remove(prep["prep"])  # the prepared sample file, no longer needed
        print(split, {**{k: prep[k] for k in ("rows", "fraud")}, **meta})
else:
    print(f"embeddings present at {paths['embeddings']} — skipping (delete the dir to re-embed)")

embeddings present at /mnt/cluster_storage/transaction-fm/embeddings/mini/ — skipping (delete the dir to re-embed)


## Check the embeddings

We load the test set's embeddings back and confirm the shape: one vector per transaction, one label per vector. The real measure of their quality is Part 6, where the fraud detectors train on them.

In [3]:
emb = np.load(os.path.join(paths["embeddings"], "embed_test.npy"))
lbl = np.load(os.path.join(paths["embeddings"], "lbl_test.npy"))

print(f"{len(emb):,} embeddings × {emb.shape[1]} dimensions   ({int(lbl.sum()):,} fraud labels)")
print(f"example embedding[:8] = {[round(float(x), 3) for x in emb[0, :8]]}  (label {int(lbl[0])})")

100,000 embeddings × 64 dimensions   (108 fraud labels)
example embedding[:8] = [-1.843, 0.358, 0.152, -1.225, -0.139, -0.797, -0.474, -0.513]  (label 0)


## Scaling factors

At `full`, this stage embeds about 1.2 million transactions in fifteen minutes: the 1M training sample plus both 100K evaluation sets. In production, every new transaction needs an embedding, so this stage runs continually.

| What grows | The limit it hits | What absorbs it | Measured at `full` |
|---|---|---|---|
| Transactions to embed | GPU-pool throughput | More GPU actors — `num_workers` in the config | ~1.2M transactions in about 15 minutes |
| CPU tokenization | CPU-pool throughput | More CPU workers | — |
| Model size | GPU memory per forward pass | Bigger batches | 512-dim embeddings at batch 256 |

Ten times the transactions would take the same GPU pool about two and a half hours; ten times the actors would bring it back to fifteen minutes. `num_workers` is the only line that changes. Memory is easy here — inference keeps no gradients or optimizer state — so each actor runs large batches.

## Takeaways

We embedded every split and wrote the results to shared storage — `embed_`, `lbl_`, and `raw_` files per split. Part 6 trains fraud detectors on these files directly.

One streaming Ray Data pipeline did the work. CPU tasks tokenized, GPU actors ran the forward passes, and each pool scales on its own. The outputs are verified against a single-GPU reference run (`scripts/verify_distributed_embed.py`).

---

## Next

**Part 6 — Fraud detectors**: train XGBoost on the raw fields, on these embeddings, and on both together — and compare against NVIDIA's published results.